In [1]:
import numpy as np 
import pandas as pd
from scipy import stats
from numpy import random
from pathlib import Path
# from statsmodels.distributions.copula.api import GumbelCopula

In [2]:
# utility functions

def make_pos_def(corr):
    eigvals, eigvecs = np.linalg.eigh(corr)
    eigvals[eigvals < 1e-8] = 1e-8  
    corr_pd = eigvecs @ np.diag(eigvals) @ eigvecs.T

    # normalize diagonal to 1
    d = np.sqrt(np.diag(corr_pd))
    corr_pd = corr_pd / d[:, None] / d[None, :]
    return corr_pd

def beta_GC(R, n, a,b,rng=None):
    R_pos = make_pos_def(R)
    mean = np.zeros(np.shape(R_pos)[0])
    if rng is None:
        z = np.random.multivariate_normal(mean, R_pos, size=n)
    else:
        z = rng.multivariate_normal(mean, R_pos, size=n)
    u = stats.norm.cdf(z)
    data = np.zeros_like(u)
    for i in range(u.shape[1]):
        data[:, i] = stats.beta.ppf(u[:, i], a[i], b[i])
    return data, R_pos

def gamma_GC(R, n, shape, scale, rng=None):
    R_pos = make_pos_def(R)
    mean = np.zeros(np.shape(R_pos)[0])
    if rng is None:
        z = np.random.multivariate_normal(mean, R_pos, size=n)
    else:
        z = rng.multivariate_normal(mean, R_pos, size=n)
    u = stats.norm.cdf(z)
    data = np.zeros_like(u)
    for i in range(u.shape[1]):
        data[:, i] = stats.gamma.ppf(u[:, i], a=shape[i], scale=scale[i])
    return data, R_pos

# def beta_GC_nonlinear(n, a, b, theta, nobj, rng):
#     copula = GumbelCopula(theta=theta, k_dim=nobj+1)
#     u = copula.rvs(nobs=n, random_state=rng)
#     data = np.zeros_like(u)
#     for i in range(u.shape[1]):
#         data[:, i] = stats.beta.ppf(u[:, i], a[i], b[i])
#     return data

def cleanupsamples(samples, nobj, precision=1):
    """Clean up samples by rounding and removing duplicates."""
    samples = np.round(samples, precision)
    c, i = np.unique(samples[:, :nobj], axis=0, return_index=True)
    newsamples = samples[i, :]  # note - these have been sorted into increasing magnitude
    if precision == 0:
        newsamples = np.array(newsamples, dtype=np.int32)
    return newsamples

def generate_beta_example_data(r, a, b, max_val, nobj, n_items=100, seed=1124, precision=0):
    item_rng = random.default_rng(seed=seed)
    
    batch = max(5, n_items // 10)
    uniq = set()
    items = []
    while len(items) < n_items:
        new, rpos = beta_GC(r, batch, a, b, rng=item_rng)
        new = 10 + new * (max_val-1)
        new = cleanupsamples(new, nobj=nobj, precision=precision)
        for item in new:
            key = tuple(item[:nobj])
            if key not in uniq:
                uniq.add(key)
                items.append(item)
                if len(items) == n_items:
                    break
    return np.unique(np.array(items), axis=0), rpos # here np.unique is used for sorting

def generate_gamma_example_data(r, shape, scale, nobj, n_items=100, seed=1124, precision=0):
    item_rng = random.default_rng(seed=seed)

    batch = max(5, n_items // 10)
    uniq = set()
    items = []
    while len(items) < n_items:
        new, rpos = gamma_GC(r, batch, shape, scale, rng=item_rng)
        new = cleanupsamples(new, nobj=nobj, precision=precision)
        for item in new:
            key = tuple(item[:nobj])
            if key not in uniq:
                uniq.add(key)
                items.append(item)
                if len(items) == n_items:
                    break
    return np.unique(np.array(items), axis=0), rpos  # here np.unique is used for sorting

# def generate_beta_example_data_nonlinear(theta, a, b, max_val, nobj, n_items=100, seed=1124, precision=0):
#     item_rng = random.default_rng(seed=seed)
    
#     batch = max(5, n_items // 10)
#     uniq = set()
#     items = []
#     while len(items) < n_items:
#         new = beta_GC_nonlinear(batch, a, b, theta, nobj, item_rng)
#         new = 1 + new * (max_val-2)
#         new = cleanupsamples(new, nobj=nobj, precision=precision)
#         for item in new:
#             key = tuple(item[:nobj])
#             if key not in uniq:
#                 uniq.add(key)
#                 items.append(item)
#                 if len(items) == n_items:
#                     break
#     return np.unique(np.array(items), axis=0)

In [ ]:
n_obj = 3
n_items = 500
items_seed = 1125
objective_range = 20
# a = [1.5,1.2,1.3,1.6,1.4,1]
# b = [2.1,1.8,1.6,1.9,2.0,1]
a = [1.5,1.3,1.6,1]
b = [2.1,1.6,1.9,1]
r = -0.3 * np.ones((n_obj + 1, n_obj + 1)) + (1 + 0.3) * np.eye(n_obj + 1)
print("objective value corr:\n", r)
cost_corr = 0.3
r[:, n_obj] = cost_corr
r[n_obj, :] = cost_corr
r[n_obj, n_obj] = 1
print("full corr:\n",r)

In [ ]:
items, rpos = generate_beta_example_data(r, a, b, max_val=objective_range, nobj=n_obj, n_items=n_items, seed=items_seed)
itemsdf = pd.DataFrame(items)
output_path = Path("data/items")
itemsdf.to_csv(output_path / f'items_obj{n_obj}_seed{items_seed}.csv', index=False, header=False)

In [ ]:
print("positive definite corr:\n", rpos)

In [ ]:
print("items corr:\n", np.round(np.corrcoef(items.T), 2))

In [ ]:
# # generate nonlinear betas

# n_obj = 5
# n_items = 60
# items_seed = 1125
# objective_range = 20
# a = [1.5,1.2,1.3,1.6,1.4,1]
# b = [2.1,1.8,1.6,1.9,2.0,1]
# theta = 2.0

# items = generate_beta_example_data_nonlinear(theta, a, b, objective_range, n_obj, n_items=n_items, seed=items_seed)
# itemsdf = pd.DataFrame(items)
# output_path = Path("data/items")
# itemsdf.to_csv(output_path / f'items_nonlinear_obj{n_obj}_seed{items_seed}.csv', index=False, header=False)

In [ ]:
# varying types of correlations between objectives
import itertools
import seaborn as sns
import matplotlib.pyplot as plt

shapes = [0.2, 0.5, 1, 2, 5, 10, 20, 50]
parameter_pairs = [
    {
        "alpha_1": a1,
        "theta_1": 1 / a1,
        "alpha_2": a2,
        "theta_2": 1 / a2,
    }
    for a1, a2 in itertools.product(shapes, repeat=2)
]

In [3]:
n_obj = 2
n_items = 500
items_seed = 1125
r = -0.3 * np.ones((n_obj + 1, n_obj + 1)) + (1 + 0.3) * np.eye(n_obj + 1)
cost_corr = 0.3
r[:, n_obj] = cost_corr
r[n_obj, :] = cost_corr
r[n_obj, n_obj] = 1
print("full corr:\n",r)

full corr:
 [[ 1.  -0.3  0.3]
 [-0.3  1.   0.3]
 [ 0.3  0.3  1. ]]


In [ ]:
from matplotlib.gridspec import GridSpec

cols = ["obj1", "obj2", "cost"]
n_vars = len(cols)
n_shapes = len(shapes)

fig = plt.figure(figsize=(3.5 * n_shapes, 3.5 * n_shapes))
outer = GridSpec(n_shapes, n_shapes, figure=fig, wspace=0.4, hspace=0.55)

for row_i, a1 in enumerate(shapes):
    for col_j, a2 in enumerate(shapes):
        theta1, theta2 = 1 / a1, 1 / a2
        a = [a1, a2, 50]
        theta = [theta1, theta2, 0.02]
        # precision=0 rounds to ints; many (α,θ) pairs then can't reach n_items
        # unique objective tuples, so the sampler loops forever.
        items, rpos = generate_gamma_example_data(
            r, a, theta, n_obj, n_items=n_items, seed=items_seed, precision=1
        )
        itemsdf = pd.DataFrame(items, columns=cols)
        print("obj 1 params:", a1, theta1, ";obj 2 params:", a2, theta2, ";max:", np.max(items, axis=0), ";min:", np.min(items, axis=0), ";median:", np.median(items, axis=0), ";range:", np.max(items, axis=0) - np.min(items, axis=0))

#         inner = outer[row_i, col_j].subgridspec(n_vars, n_vars, wspace=0.08, hspace=0.08)
#         axes = np.empty((n_vars, n_vars), dtype=object)
#         for i, yi in enumerate(cols):
#             for j, xj in enumerate(cols):
#                 ax = fig.add_subplot(inner[i, j])
#                 axes[i, j] = ax
#                 if i == j:
#                     ax.hist(itemsdf[xj], bins=8, color="C0", alpha=0.75)
#                 else:
#                     ax.scatter(itemsdf[xj], itemsdf[yi], s=6, alpha=0.65, color="C0")
#                 ax.tick_params(labelbottom=False, labelleft=False, length=0)
#                 if i == n_vars - 1:
#                     ax.set_xlabel(xj, fontsize=6)
#                     ax.tick_params(labelbottom=True, labelsize=5, length=2)
#                 if j == 0:
#                     ax.set_ylabel(yi, fontsize=6)
#                     ax.tick_params(labelleft=True, labelsize=5, length=2)

#         axes[0, 1].set_title(
#             f"$\\alpha_1$={a1}, $\\theta_1$={theta1:.3g}\n"
#             f"$\\alpha_2$={a2}, $\\theta_2$={theta2:.3g}",
#             fontsize=7,
#             pad=4,
#         )

# fig.suptitle("Pairplots by gamma parameters (rows: obj1, cols: obj2)", fontsize=14, y=0.995)
# fig.savefig(
#     "pairplots_by_gamma_params.png",
#     dpi=150,
#     bbox_inches="tight",
# )
# plt.show()

In [ ]:
itemsdf

In [4]:
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
import itertools

beta_shapes = [
    (0.2, 0.2),
    (0.5, 0.5),
    (0.5, 1.0),
    (1.0, 0.5),
    (0.5, 2.0),
    (2.0, 0.5),
    (1.0, 1.0),
    (1.0, 2.0),
    (2.0, 1.0),
    (2.0, 2.0),
    (2.0, 5.0),
    (5.0, 2.0),
    (5.0, 5.0),
    (10.0, 10.0),
    (50.0, 50.0),
]

parameter_pairs = [
    {
        "alpha_1": alpha_1,
        "beta_1": beta_1,
        "alpha_2": alpha_2,
        "beta_2": beta_2,
    }
    for (alpha_1, beta_1), (alpha_2, beta_2) in itertools.product(beta_shapes, repeat=2)
]
assert len(parameter_pairs) == 225, f"expected 225 pairs, got {len(parameter_pairs)}"

cols = ["obj1", "obj2", "cost"]
n_vars = len(cols)
n_shapes = len(beta_shapes)
# Broad bell shape
# 1, 1 would be uniform
a_cost, b_cost = 2.0, 2.0
objective_range = 90

fig = plt.figure(figsize=(3.5 * n_shapes, 3.5 * n_shapes))
outer = GridSpec(n_shapes, n_shapes, figure=fig, wspace=0.4, hspace=0.55)

items_dict = {}
itemsdf_random_dict = {}
for row_i, (alpha_1, beta_1) in enumerate(beta_shapes):
    for col_j, (alpha_2, beta_2) in enumerate(beta_shapes):
        a = [alpha_1, alpha_2, a_cost]
        b = [beta_1, beta_2, b_cost]
        # precision=0 rounds to ints; many (α,β) pairs then can't reach n_items
        # unique objective tuples, so the sampler loops forever.
        items, rpos = generate_beta_example_data(
            r,
            a,
            b,
            max_val=objective_range,
            nobj=n_obj,
            n_items=n_items,
            seed=items_seed,
            precision=0,
        )
        itemsdf = pd.DataFrame(items, columns=cols)
        print("obj 1 params:", alpha_1, beta_1, ";obj 2 params:", alpha_2, beta_2, ";max:", np.max(items, axis=0), ";min:", np.min(items, axis=0), ";median:", np.median(items, axis=0), ";range:", np.max(items, axis=0) - np.min(items, axis=0))
        items_dict[(alpha_1, beta_1, alpha_2, beta_2)] = items
        itemsdf_random_dict[(alpha_1, beta_1, alpha_2, beta_2)] = itemsdf.sample(frac=1, random_state=42).reset_index(drop=True)
#         inner = outer[row_i, col_j].subgridspec(n_vars, n_vars, wspace=0.08, hspace=0.08)
#         axes = np.empty((n_vars, n_vars), dtype=object)
#         for i, yi in enumerate(cols):
#             for j, xj in enumerate(cols):
#                 ax = fig.add_subplot(inner[i, j])
#                 axes[i, j] = ax
#                 if i == j:
#                     ax.hist(itemsdf[xj], bins=8, color="C0", alpha=0.75)
#                 else:
#                     ax.scatter(itemsdf[xj], itemsdf[yi], s=6, alpha=0.65, color="C0")
#                 ax.tick_params(labelbottom=False, labelleft=False, length=0)
#                 if i == n_vars - 1:
#                     ax.set_xlabel(xj, fontsize=6)
#                     ax.tick_params(labelbottom=True, labelsize=5, length=2)
#                 if j == 0:
#                     ax.set_ylabel(yi, fontsize=6)
#                     ax.tick_params(labelleft=True, labelsize=5, length=2)

#         axes[0, 1].set_title(
#             f"$\\alpha_1$={alpha_1}, $\\beta_1$={beta_1}\n"
#             f"$\\alpha_2$={alpha_2}, $\\beta_2$={beta_2}",
#             fontsize=7,
#             pad=4,
#         )

# fig.suptitle("Pairplots by beta parameters (rows: obj1, cols: obj2)", fontsize=14, y=0.995)
# fig.savefig(
#     "pairplots_by_beta_params.png",
#     dpi=150,
#     bbox_inches="tight",
# )
# plt.show()


obj 1 params: 0.2 0.2 ;obj 2 params: 0.2 0.2 ;max: [99 99 95] ;min: [10 10 13] ;median: [52.5 50.5 54. ] ;range: [89 89 82]
obj 1 params: 0.2 0.2 ;obj 2 params: 0.5 0.5 ;max: [99 99 95] ;min: [10 10 13] ;median: [56. 51. 54.] ;range: [89 89 82]
obj 1 params: 0.2 0.2 ;obj 2 params: 0.5 1.0 ;max: [99 99 96] ;min: [10 10 13] ;median: [50.5 37.  55. ] ;range: [89 89 83]
obj 1 params: 0.2 0.2 ;obj 2 params: 1.0 0.5 ;max: [99 99 95] ;min: [10 11 13] ;median: [58.5 70.  54. ] ;range: [89 88 82]
obj 1 params: 0.2 0.2 ;obj 2 params: 0.5 2.0 ;max: [99 96 98] ;min: [10 10 13] ;median: [47. 24. 55.] ;range: [89 86 85]
obj 1 params: 0.2 0.2 ;obj 2 params: 2.0 0.5 ;max: [99 99 95] ;min: [10 23 13] ;median: [59.5 83.  54. ] ;range: [89 76 82]
obj 1 params: 0.2 0.2 ;obj 2 params: 1.0 1.0 ;max: [99 99 95] ;min: [10 11 13] ;median: [50.5 52.  54. ] ;range: [89 88 82]
obj 1 params: 0.2 0.2 ;obj 2 params: 1.0 2.0 ;max: [99 97 98] ;min: [10 10 13] ;median: [53.5 36.  55. ] ;range: [89 87 85]
obj 1 params: 

<Figure size 5250x5250 with 0 Axes>

In [5]:
itemsdf_random_dict[(0.5, 0.5, 2.0, 1.0)]

,obj1,obj2,cost
0,78,45,71
1,16,43,19
2,81,39,58
3,30,89,61
4,20,99,44
...,...,...,...
495,21,72,66
496,59,55,44
497,75,87,86
498,93,71,44


In [ ]:
a, b = 50.0, 50.0
x = np.linspace(0, 1, 300)
pdf = stats.beta.pdf(x, a, b)
plt.plot(x, pdf)
plt.xlabel("x")
plt.ylabel("pdf")
plt.title(rf"Beta(a={a}, b={b})")
plt.show()